In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

SEED = 42

TARGET_INTENTS = [
    "Reveal",
    "Interpret",
    "Revise",
    "Chitchat",
    "Inquire",
]

N_FOLDS = 5

random.seed(SEED)
np.random.seed(SEED)

print("Seed:", SEED)
print("Target intents:", TARGET_INTENTS)
print("Number of folds:", N_FOLDS)

Seed: 42
Target intents: ['Reveal', 'Interpret', 'Revise', 'Chitchat', 'Inquire']
Number of folds: 5


In [3]:
DATA_PATH = "/content/drive/MyDrive/conversation_en.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print("Raw conversations:", len(raw_data))

Raw conversations: 826


In [4]:
conv_ids = [conv["conv_id"] for conv in raw_data]

print("Total conversations:", len(conv_ids))
print("Unique conversation IDs:", len(set(conv_ids)))

duplicate_ids = [
    conv_id
    for conv_id, count in Counter(conv_ids).items()
    if count > 1
]

print("Duplicate conversation IDs:", duplicate_ids)

assert len(conv_ids) == len(set(conv_ids)), \
    "Duplicate conversation IDs found."

Total conversations: 826
Unique conversation IDs: 826
Duplicate conversation IDs: []


In [5]:
conversation_summary = []

for conv in raw_data:
    conv_id = conv["conv_id"]

    counts = {
        intent: 0
        for intent in TARGET_INTENTS
    }

    # Count target USER turns only
    for msg in conv["conversation"]:

        if msg.get("role") != "user":
            continue

        intent = msg.get("intent")

        if intent in TARGET_INTENTS:
            counts[intent] += 1

    total_5class = sum(counts.values())

    conversation_summary.append({
        "conv_id": conv_id,
        "Reveal": counts["Reveal"],
        "Interpret": counts["Interpret"],
        "Revise": counts["Revise"],
        "Chitchat": counts["Chitchat"],
        "Inquire": counts["Inquire"],
        "total_5class": total_5class,
    })

audit_df = pd.DataFrame(conversation_summary)

print("Conversations in audit table:", len(audit_df))

display(audit_df.head(10))

Conversations in audit table: 826


,conv_id,Reveal,Interpret,Revise,Chitchat,Inquire,total_5class
0,19762,1,7,2,0,0,10
1,20322,1,4,0,1,0,6
2,20534,1,6,0,0,0,7
3,20771,1,7,0,1,0,9
4,21152,1,1,0,1,0,3
5,21443,1,6,0,0,0,7
6,21951,1,2,0,0,0,3
7,22766,7,0,0,0,0,7
8,23826,1,1,0,1,1,4
9,24741,1,1,0,1,0,3


In [6]:
eligible_df = audit_df[
    audit_df["total_5class"] > 0
].copy().reset_index(drop=True)

excluded_df = audit_df[
    audit_df["total_5class"] == 0
].copy().reset_index(drop=True)

print("Eligible 5-class conversations:", len(eligible_df))
print("Excluded conversations:", len(excluded_df))

if len(excluded_df) > 0:
    display(excluded_df)

Eligible 5-class conversations: 826
Excluded conversations: 0


In [7]:
class_totals = eligible_df[TARGET_INTENTS].sum()

print("5-class totals:")
print(class_totals)

print("\nTotal labeled user turns:", class_totals.sum())

print("\nNumber of conversations containing each intent:")
for intent in TARGET_INTENTS:
    n_convs = (eligible_df[intent] > 0).sum()
    print(f"{intent}: {n_convs} conversations")

5-class totals:
Reveal       1031
Interpret    1813
Revise         67
Chitchat      268
Inquire        17
dtype: int64

Total labeled user turns: 3196

Number of conversations containing each intent:
Reveal: 778 conversations
Interpret: 665 conversations
Revise: 53 conversations
Chitchat: 212 conversations
Inquire: 13 conversations


In [8]:
N_CONVERSATIONS = len(eligible_df)

base_fold_size = N_CONVERSATIONS // N_FOLDS
remainder = N_CONVERSATIONS % N_FOLDS

fold_sizes = [
    base_fold_size + (1 if i < remainder else 0)
    for i in range(N_FOLDS)
]

print("Eligible conversations:", N_CONVERSATIONS)
print("Number of folds:", N_FOLDS)
print("Fold sizes:", fold_sizes)
print("Total:", sum(fold_sizes))

assert sum(fold_sizes) == N_CONVERSATIONS
assert max(fold_sizes) - min(fold_sizes) <= 1

Eligible conversations: 826
Number of folds: 5
Fold sizes: [166, 165, 165, 165, 165]
Total: 826


In [9]:
# Verify total class counts before any shuffle/split

class_totals = eligible_df[TARGET_INTENTS].sum()

print("5-class totals:")
print(class_totals)

print("\nTotal labeled user turns:", class_totals.sum())

5-class totals:
Reveal       1031
Interpret    1813
Revise         67
Chitchat      268
Inquire        17
dtype: int64

Total labeled user turns: 3196


In [10]:
# ONE-TIME deterministic shuffle of all eligible 5-class conversations

shuffled_df = eligible_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

# Preserve this position permanently for deterministic tie-breaking
shuffled_df["shuffle_order"] = np.arange(len(shuffled_df))

print("Number of shuffled conversations:", len(shuffled_df))

print("\nFirst 20 shuffled conversation IDs:")
print(shuffled_df["conv_id"].head(20).tolist())

Number of shuffled conversations: 826

First 20 shuffled conversation IDs:
[108393, 7487, 37857, 6508, 61169, 68834, 129429, 42246, 94924, 117490, 94309, 59854, 128244, 28400, 74517, 111226, 31061, 14412, 112849, 72507]


In [11]:
folds = {
    fold_id: []
    for fold_id in range(1, N_FOLDS + 1)
}

fold_class_counts = {
    fold_id: {
        intent: 0
        for intent in TARGET_INTENTS
    }
    for fold_id in range(1, N_FOLDS + 1)
}

fold_capacity = {
    fold_id: fold_sizes[fold_id - 1]
    for fold_id in range(1, N_FOLDS + 1)
}

print("Fold capacities:")
print(fold_capacity)

Fold capacities:
{1: 166, 2: 165, 3: 165, 4: 165, 5: 165}


In [12]:
# Target class counts per outer fold
# These are goals only — exact matching may not be possible
# because whole conversations must stay together.

target_per_fold = {
    intent: class_totals[intent] / N_FOLDS
    for intent in TARGET_INTENTS
}

print("Ideal target turns per fold:")
for intent, target in target_per_fold.items():
    print(f"{intent:10s}: {target:.2f}")

print("\nFold capacities:")
for fold_id in range(1, N_FOLDS + 1):
    print(f"Fold {fold_id}: {fold_capacity[fold_id]} conversations")

Ideal target turns per fold:
Reveal    : 206.20
Interpret : 362.60
Revise    : 13.40
Chitchat  : 53.60
Inquire   : 3.40

Fold capacities:
Fold 1: 166 conversations
Fold 2: 165 conversations
Fold 3: 165 conversations
Fold 4: 165 conversations
Fold 5: 165 conversations


In [13]:
# Inspect rare-intent conversations before allocation

rare_cols = ["Inquire", "Revise"]

for intent in rare_cols:
    temp = shuffled_df[shuffled_df[intent] > 0][
        ["conv_id", intent, "shuffle_order"]
    ].copy()

    print(f"\n{intent}-containing conversations: {len(temp)}")
    print(f"Total {intent} turns: {temp[intent].sum()}")

    display(temp)


Inquire-containing conversations: 13
Total Inquire turns: 17


,conv_id,Inquire,shuffle_order
106,59888,2,106
153,66237,3,153
283,64952,2,283
305,79028,1,305
351,57558,1,351
370,3340,1,370
376,93013,1,376
533,62214,1,533
569,3005,1,569
575,23826,1,575



Revise-containing conversations: 53
Total Revise turns: 67


,conv_id,Revise,shuffle_order
4,61169,1,4
9,117490,1,9
34,57687,4,34
78,104558,1,78
87,101020,1,87
91,82889,1,91
100,105761,1,100
105,104361,3,105
106,59888,1,106
141,116141,1,141


In [14]:
# ============================================================
# STEP 1: Allocate INQUIRE-containing conversations first
# ============================================================

assigned_conv_ids = set()

# Take Inquire conversations in the original ONE-TIME shuffled order
inquire_df = (
    shuffled_df[shuffled_df["Inquire"] > 0]
    .sort_values("shuffle_order")
    .copy()
)

for _, row in inquire_df.iterrows():

    # Only folds that still have capacity
    available_folds = [
        fold_id
        for fold_id in range(1, N_FOLDS + 1)
        if len(folds[fold_id]) < fold_capacity[fold_id]
    ]

    # Choose the fold with:
    # 1. lowest current Inquire turns
    # 2. then lowest number of conversations
    # 3. then fold ID for deterministic tie-breaking
    best_fold = min(
        available_folds,
        key=lambda f: (
            fold_class_counts[f]["Inquire"],
            len(folds[f]),
            f
        )
    )

    # Assign whole conversation
    folds[best_fold].append(row["conv_id"])
    assigned_conv_ids.add(row["conv_id"])

    # Update ALL intent counts contributed by this conversation
    for intent in TARGET_INTENTS:
        fold_class_counts[best_fold][intent] += int(row[intent])


print("Assigned Inquire-containing conversations:", len(assigned_conv_ids))

print("\nDistribution after Inquire allocation:")
for fold_id in range(1, N_FOLDS + 1):
    print(
        f"Fold {fold_id}: "
        f"conversations={len(folds[fold_id])}, "
        f"Inquire turns={fold_class_counts[fold_id]['Inquire']}, "
        f"Revise turns={fold_class_counts[fold_id]['Revise']}, "
        f"Chitchat turns={fold_class_counts[fold_id]['Chitchat']}, "
        f"Reveal turns={fold_class_counts[fold_id]['Reveal']}, "
        f"Interpret turns={fold_class_counts[fold_id]['Interpret']}"
    )

Assigned Inquire-containing conversations: 13

Distribution after Inquire allocation:
Fold 1: conversations=3, Inquire turns=4, Revise turns=2, Chitchat turns=1, Reveal turns=2, Interpret turns=3
Fold 2: conversations=2, Inquire turns=4, Revise turns=0, Chitchat turns=0, Reveal turns=4, Interpret turns=1
Fold 3: conversations=2, Inquire turns=3, Revise turns=0, Chitchat turns=0, Reveal turns=1, Interpret turns=2
Fold 4: conversations=3, Inquire turns=3, Revise turns=2, Chitchat turns=1, Reveal turns=3, Interpret turns=4
Fold 5: conversations=3, Inquire turns=3, Revise turns=2, Chitchat turns=2, Reveal turns=3, Interpret turns=6


In [15]:
# ============================================================
# STEP 2: Allocate remaining REVISE-containing conversations
# ============================================================

revise_df = (
    shuffled_df[
        (shuffled_df["Revise"] > 0) &
        (~shuffled_df["conv_id"].isin(assigned_conv_ids))
    ]
    .sort_values("shuffle_order")
    .copy()
)

for _, row in revise_df.iterrows():

    available_folds = [
        fold_id
        for fold_id in range(1, N_FOLDS + 1)
        if len(folds[fold_id]) < fold_capacity[fold_id]
    ]

    # Choose fold with:
    # 1. lowest current Revise turns
    # 2. then lowest current total rare-class load
    # 3. then lowest number of conversations
    # 4. deterministic fold ID
    best_fold = min(
        available_folds,
        key=lambda f: (
            fold_class_counts[f]["Revise"],
            fold_class_counts[f]["Inquire"] + fold_class_counts[f]["Revise"],
            len(folds[f]),
            f
        )
    )

    folds[best_fold].append(row["conv_id"])
    assigned_conv_ids.add(row["conv_id"])

    for intent in TARGET_INTENTS:
        fold_class_counts[best_fold][intent] += int(row[intent])


print("Total assigned conversations so far:", len(assigned_conv_ids))

print("\nDistribution after Revise allocation:")
for fold_id in range(1, N_FOLDS + 1):
    print(
        f"Fold {fold_id}: "
        f"conversations={len(folds[fold_id])}, "
        f"Inquire={fold_class_counts[fold_id]['Inquire']}, "
        f"Revise={fold_class_counts[fold_id]['Revise']}, "
        f"Chitchat={fold_class_counts[fold_id]['Chitchat']}, "
        f"Reveal={fold_class_counts[fold_id]['Reveal']}, "
        f"Interpret={fold_class_counts[fold_id]['Interpret']}"
    )

Total assigned conversations so far: 62

Distribution after Revise allocation:
Fold 1: conversations=12, Inquire=4, Revise=13, Chitchat=4, Reveal=12, Interpret=20
Fold 2: conversations=11, Inquire=4, Revise=13, Chitchat=5, Reveal=9, Interpret=13
Fold 3: conversations=13, Inquire=3, Revise=14, Chitchat=6, Reveal=10, Interpret=20
Fold 4: conversations=14, Inquire=3, Revise=13, Chitchat=7, Reveal=14, Interpret=22
Fold 5: conversations=12, Inquire=3, Revise=14, Chitchat=7, Reveal=11, Interpret=21


In [16]:
# ============================================================
# STEP 3: Allocate remaining CHITCHAT-containing conversations
# ============================================================

chitchat_df = (
    shuffled_df[
        (shuffled_df["Chitchat"] > 0) &
        (~shuffled_df["conv_id"].isin(assigned_conv_ids))
    ]
    .sort_values("shuffle_order")
    .copy()
)

for _, row in chitchat_df.iterrows():

    available_folds = [
        fold_id
        for fold_id in range(1, N_FOLDS + 1)
        if len(folds[fold_id]) < fold_capacity[fold_id]
    ]

    # Choose the fold with:
    # 1. lowest current Chitchat count
    # 2. then lowest overall rare-intent load
    # 3. then lowest number of conversations
    # 4. deterministic fold ID
    best_fold = min(
        available_folds,
        key=lambda f: (
            fold_class_counts[f]["Chitchat"],
            (
                fold_class_counts[f]["Inquire"]
                + fold_class_counts[f]["Revise"]
                + fold_class_counts[f]["Chitchat"]
            ),
            len(folds[f]),
            f
        )
    )

    folds[best_fold].append(row["conv_id"])
    assigned_conv_ids.add(row["conv_id"])

    for intent in TARGET_INTENTS:
        fold_class_counts[best_fold][intent] += int(row[intent])


print("Total assigned conversations so far:", len(assigned_conv_ids))

print("\nDistribution after Chitchat allocation:")
for fold_id in range(1, N_FOLDS + 1):
    print(
        f"Fold {fold_id}: "
        f"conversations={len(folds[fold_id])}, "
        f"Inquire={fold_class_counts[fold_id]['Inquire']}, "
        f"Revise={fold_class_counts[fold_id]['Revise']}, "
        f"Chitchat={fold_class_counts[fold_id]['Chitchat']}, "
        f"Reveal={fold_class_counts[fold_id]['Reveal']}, "
        f"Interpret={fold_class_counts[fold_id]['Interpret']}"
    )

Total assigned conversations so far: 255

Distribution after Chitchat allocation:
Fold 1: conversations=52, Inquire=4, Revise=13, Chitchat=53, Reveal=71, Interpret=80
Fold 2: conversations=48, Inquire=4, Revise=13, Chitchat=54, Reveal=55, Interpret=73
Fold 3: conversations=52, Inquire=3, Revise=14, Chitchat=53, Reveal=52, Interpret=101
Fold 4: conversations=51, Inquire=3, Revise=13, Chitchat=54, Reveal=59, Interpret=72
Fold 5: conversations=52, Inquire=3, Revise=14, Chitchat=54, Reveal=56, Interpret=102


In [17]:
# ============================================================
# STEP 4: Allocate ALL remaining conversations
# Balance Reveal + Interpret + fold capacity
# ============================================================

remaining_df = (
    shuffled_df[
        ~shuffled_df["conv_id"].isin(assigned_conv_ids)
    ]
    .sort_values("shuffle_order")
    .copy()
)

print("Remaining conversations to assign:", len(remaining_df))


for _, row in remaining_df.iterrows():

    available_folds = [
        fold_id
        for fold_id in range(1, N_FOLDS + 1)
        if len(folds[fold_id]) < fold_capacity[fold_id]
    ]

    # Score each available fold based on how far it would be
    # from the ideal per-fold class targets after adding this conversation.
    def fold_score(f):

        new_reveal = (
            fold_class_counts[f]["Reveal"]
            + int(row["Reveal"])
        )

        new_interpret = (
            fold_class_counts[f]["Interpret"]
            + int(row["Interpret"])
        )

        # normalized distance from ideal targets
        reveal_error = abs(
            new_reveal - target_per_fold["Reveal"]
        ) / target_per_fold["Reveal"]

        interpret_error = abs(
            new_interpret - target_per_fold["Interpret"]
        ) / target_per_fold["Interpret"]

        # encourage fold-size balance
        size_after = len(folds[f]) + 1
        size_ratio = size_after / fold_capacity[f]

        return (
            reveal_error + interpret_error,
            size_ratio,
            f
        )

    best_fold = min(
        available_folds,
        key=fold_score
    )

    folds[best_fold].append(row["conv_id"])
    assigned_conv_ids.add(row["conv_id"])

    for intent in TARGET_INTENTS:
        fold_class_counts[best_fold][intent] += int(row[intent])


print("\nTotal assigned conversations:", len(assigned_conv_ids))

print("\nFINAL OUTER FOLD DISTRIBUTION:")
for fold_id in range(1, N_FOLDS + 1):

    total_turns = sum(
        fold_class_counts[fold_id][intent]
        for intent in TARGET_INTENTS
    )

    print(
        f"Fold {fold_id}: "
        f"conversations={len(folds[fold_id])}, "
        f"Inquire={fold_class_counts[fold_id]['Inquire']}, "
        f"Revise={fold_class_counts[fold_id]['Revise']}, "
        f"Chitchat={fold_class_counts[fold_id]['Chitchat']}, "
        f"Reveal={fold_class_counts[fold_id]['Reveal']}, "
        f"Interpret={fold_class_counts[fold_id]['Interpret']}, "
        f"Total turns={total_turns}"
    )

Remaining conversations to assign: 571

Total assigned conversations: 826

FINAL OUTER FOLD DISTRIBUTION:
Fold 1: conversations=166, Inquire=4, Revise=13, Chitchat=53, Reveal=215, Interpret=375, Total turns=660
Fold 2: conversations=165, Inquire=4, Revise=13, Chitchat=54, Reveal=212, Interpret=335, Total turns=618
Fold 3: conversations=165, Inquire=3, Revise=14, Chitchat=53, Reveal=199, Interpret=376, Total turns=645
Fold 4: conversations=165, Inquire=3, Revise=13, Chitchat=54, Reveal=209, Interpret=338, Total turns=617
Fold 5: conversations=165, Inquire=3, Revise=14, Chitchat=54, Reveal=196, Interpret=389, Total turns=656


In [18]:
# ============================================================
# STEP 5: Refine Reveal / Interpret balance using safe swaps
# ------------------------------------------------------------
# IMPORTANT:
# Only swap conversations containing NONE of:
# Inquire, Revise, Chitchat
#
# Therefore rare-class distributions remain unchanged.
# Fold sizes also remain unchanged because swaps are 1-for-1.
# ============================================================

row_lookup = (
    shuffled_df
    .set_index("conv_id")
    .to_dict("index")
)

target_reveal = target_per_fold["Reveal"]
target_interpret = target_per_fold["Interpret"]


def global_balance_score():
    """
    Lower = better.
    Measures normalized squared deviation from the ideal
    Reveal and Interpret counts across all 5 folds.
    """
    score = 0.0

    for f in range(1, N_FOLDS + 1):

        r = fold_class_counts[f]["Reveal"]
        i = fold_class_counts[f]["Interpret"]

        score += ((r - target_reveal) / target_reveal) ** 2
        score += ((i - target_interpret) / target_interpret) ** 2

    return score


def is_safe_swap_conv(conv_id):
    row = row_lookup[conv_id]

    return (
        int(row["Inquire"]) == 0
        and int(row["Revise"]) == 0
        and int(row["Chitchat"]) == 0
    )


print("Balance score BEFORE refinement:",
      global_balance_score())


MAX_PASSES = 20

for pass_num in range(MAX_PASSES):

    best_improvement = 0.0
    best_swap = None
    current_score = global_balance_score()

    # Compare every pair of folds
    for f1 in range(1, N_FOLDS + 1):

        candidates1 = [
            cid
            for cid in folds[f1]
            if is_safe_swap_conv(cid)
        ]

        for f2 in range(f1 + 1, N_FOLDS + 1):

            candidates2 = [
                cid
                for cid in folds[f2]
                if is_safe_swap_conv(cid)
            ]

            for cid1 in candidates1:
                row1 = row_lookup[cid1]

                for cid2 in candidates2:
                    row2 = row_lookup[cid2]

                    # Current counts
                    r1 = fold_class_counts[f1]["Reveal"]
                    i1 = fold_class_counts[f1]["Interpret"]

                    r2 = fold_class_counts[f2]["Reveal"]
                    i2 = fold_class_counts[f2]["Interpret"]

                    # Counts after hypothetical swap
                    new_r1 = (
                        r1
                        - int(row1["Reveal"])
                        + int(row2["Reveal"])
                    )

                    new_i1 = (
                        i1
                        - int(row1["Interpret"])
                        + int(row2["Interpret"])
                    )

                    new_r2 = (
                        r2
                        - int(row2["Reveal"])
                        + int(row1["Reveal"])
                    )

                    new_i2 = (
                        i2
                        - int(row2["Interpret"])
                        + int(row1["Interpret"])
                    )

                    # Score for only these two folds BEFORE
                    old_pair_score = (
                        ((r1 - target_reveal) / target_reveal) ** 2
                        + ((i1 - target_interpret) / target_interpret) ** 2
                        + ((r2 - target_reveal) / target_reveal) ** 2
                        + ((i2 - target_interpret) / target_interpret) ** 2
                    )

                    # Score AFTER swap
                    new_pair_score = (
                        ((new_r1 - target_reveal) / target_reveal) ** 2
                        + ((new_i1 - target_interpret) / target_interpret) ** 2
                        + ((new_r2 - target_reveal) / target_reveal) ** 2
                        + ((new_i2 - target_interpret) / target_interpret) ** 2
                    )

                    improvement = old_pair_score - new_pair_score

                    if improvement > best_improvement:
                        best_improvement = improvement
                        best_swap = (
                            f1, cid1,
                            f2, cid2
                        )

    # No improving swap remains
    if best_swap is None:
        print(
            f"Stopped after {pass_num} refinement passes."
        )
        break

    f1, cid1, f2, cid2 = best_swap

    row1 = row_lookup[cid1]
    row2 = row_lookup[cid2]

    # Swap conversation IDs
    idx1 = folds[f1].index(cid1)
    idx2 = folds[f2].index(cid2)

    folds[f1][idx1] = cid2
    folds[f2][idx2] = cid1

    # Update all class counts
    for intent in TARGET_INTENTS:

        fold_class_counts[f1][intent] += (
            int(row2[intent])
            - int(row1[intent])
        )

        fold_class_counts[f2][intent] += (
            int(row1[intent])
            - int(row2[intent])
        )


print(
    "\nBalance score AFTER refinement:",
    global_balance_score()
)

print("\nREFINED OUTER FOLD DISTRIBUTION:")

for fold_id in range(1, N_FOLDS + 1):

    total_turns = sum(
        fold_class_counts[fold_id][intent]
        for intent in TARGET_INTENTS
    )

    print(
        f"Fold {fold_id}: "
        f"conversations={len(folds[fold_id])}, "
        f"Inquire={fold_class_counts[fold_id]['Inquire']}, "
        f"Revise={fold_class_counts[fold_id]['Revise']}, "
        f"Chitchat={fold_class_counts[fold_id]['Chitchat']}, "
        f"Reveal={fold_class_counts[fold_id]['Reveal']}, "
        f"Interpret={fold_class_counts[fold_id]['Interpret']}, "
        f"Total={total_turns}"
    )

Balance score BEFORE refinement: 0.02469568087870138
Stopped after 14 refinement passes.

Balance score AFTER refinement: 2.7942315234978408e-05

REFINED OUTER FOLD DISTRIBUTION:
Fold 1: conversations=166, Inquire=4, Revise=13, Chitchat=53, Reveal=207, Interpret=363, Total=640
Fold 2: conversations=165, Inquire=4, Revise=13, Chitchat=54, Reveal=206, Interpret=362, Total=639
Fold 3: conversations=165, Inquire=3, Revise=14, Chitchat=53, Reveal=206, Interpret=362, Total=638
Fold 4: conversations=165, Inquire=3, Revise=13, Chitchat=54, Reveal=206, Interpret=363, Total=639
Fold 5: conversations=165, Inquire=3, Revise=14, Chitchat=54, Reveal=206, Interpret=363, Total=640


In [19]:
# ============================================================
# FINAL 5-CLASS OUTER-FOLD VERIFICATION
# ============================================================

all_ids = []

print("FINAL OUTER FOLD SUMMARY\n")

for fold_id in range(1, N_FOLDS + 1):

    fold_ids = folds[fold_id]
    all_ids.extend(fold_ids)

    counts = fold_class_counts[fold_id]

    total_turns = sum(counts[intent] for intent in TARGET_INTENTS)

    print(
        f"Fold {fold_id}: "
        f"Conversations={len(fold_ids)}, "
        f"Reveal={counts['Reveal']}, "
        f"Interpret={counts['Interpret']}, "
        f"Revise={counts['Revise']}, "
        f"Chitchat={counts['Chitchat']}, "
        f"Inquire={counts['Inquire']}, "
        f"TotalTurns={total_turns}"
    )

# ------------------------------------------------------------
# 1. Every conversation assigned exactly once
# ------------------------------------------------------------

print("\nConversation assignment audit:")
print("Total assigned IDs:", len(all_ids))
print("Unique assigned IDs:", len(set(all_ids)))

assert len(all_ids) == 826
assert len(set(all_ids)) == 826


# ------------------------------------------------------------
# 2. Expected fold capacities
# ------------------------------------------------------------

observed_sizes = [
    len(folds[f])
    for f in range(1, N_FOLDS + 1)
]

print("\nObserved fold sizes:", observed_sizes)
print("Expected fold sizes:", fold_sizes)

assert sorted(observed_sizes) == sorted(fold_sizes)


# ------------------------------------------------------------
# 3. Pairwise fold overlap
# ------------------------------------------------------------

print("\nPairwise fold overlaps:")

for i in range(1, N_FOLDS + 1):
    for j in range(i + 1, N_FOLDS + 1):

        overlap = set(folds[i]) & set(folds[j])

        print(f"F{i} ∩ F{j}: {len(overlap)}")

        assert len(overlap) == 0


# ------------------------------------------------------------
# 4. Reconstruct global class totals from all folds
# ------------------------------------------------------------

reconstructed_totals = {
    intent: sum(
        fold_class_counts[f][intent]
        for f in range(1, N_FOLDS + 1)
    )
    for intent in TARGET_INTENTS
}

print("\nReconstructed class totals:")

for intent in TARGET_INTENTS:
    print(
        f"{intent}: "
        f"{reconstructed_totals[intent]} "
        f"(expected {int(class_totals[intent])})"
    )

    assert reconstructed_totals[intent] == int(class_totals[intent])


# ------------------------------------------------------------
# 5. Check rare-class coverage in EVERY test fold
# ------------------------------------------------------------

print("\nRare-class coverage:")

for fold_id in range(1, N_FOLDS + 1):

    inquire_n = fold_class_counts[fold_id]["Inquire"]
    revise_n = fold_class_counts[fold_id]["Revise"]

    print(
        f"Fold {fold_id}: "
        f"Inquire={inquire_n}, "
        f"Revise={revise_n}"
    )

    assert inquire_n > 0
    assert revise_n > 0


print("\nPASS: 5-class outer folds are valid, mutually disjoint,")
print("cover all 826 conversations exactly once,")
print("reconstruct all 3,196 target turns,")
print("and contain both Inquire and Revise in every test fold.")

FINAL OUTER FOLD SUMMARY

Fold 1: Conversations=166, Reveal=207, Interpret=363, Revise=13, Chitchat=53, Inquire=4, TotalTurns=640
Fold 2: Conversations=165, Reveal=206, Interpret=362, Revise=13, Chitchat=54, Inquire=4, TotalTurns=639
Fold 3: Conversations=165, Reveal=206, Interpret=362, Revise=14, Chitchat=53, Inquire=3, TotalTurns=638
Fold 4: Conversations=165, Reveal=206, Interpret=363, Revise=13, Chitchat=54, Inquire=3, TotalTurns=639
Fold 5: Conversations=165, Reveal=206, Interpret=363, Revise=14, Chitchat=54, Inquire=3, TotalTurns=640

Conversation assignment audit:
Total assigned IDs: 826
Unique assigned IDs: 826

Observed fold sizes: [166, 165, 165, 165, 165]
Expected fold sizes: [166, 165, 165, 165, 165]

Pairwise fold overlaps:
F1 ∩ F2: 0
F1 ∩ F3: 0
F1 ∩ F4: 0
F1 ∩ F5: 0
F2 ∩ F3: 0
F2 ∩ F4: 0
F2 ∩ F5: 0
F3 ∩ F4: 0
F3 ∩ F5: 0
F4 ∩ F5: 0

Reconstructed class totals:
Reveal: 1031 (expected 1031)
Interpret: 1813 (expected 1813)
Revise: 67 (expected 67)
Chitchat: 268 (expected 268)

In [20]:
# ============================================================
# SAVE FROZEN 5-CLASS OUTER FOLDS
# ============================================================

import os
import pandas as pd

SAVE_DIR = "/content/drive/MyDrive/5class_5cv_frozen"
os.makedirs(SAVE_DIR, exist_ok=True)

for fold_id in range(1, N_FOLDS + 1):

    fold_df = pd.DataFrame({
        "conversation_id": folds[fold_id]
    })

    path = os.path.join(
        SAVE_DIR,
        f"fold{fold_id}_test_ids.csv"
    )

    fold_df.to_csv(path, index=False)

    print(
        f"Saved Fold {fold_id}: "
        f"{len(fold_df)} conversations -> {path}"
    )

Saved Fold 1: 166 conversations -> /content/drive/MyDrive/5class_5cv_frozen/fold1_test_ids.csv
Saved Fold 2: 165 conversations -> /content/drive/MyDrive/5class_5cv_frozen/fold2_test_ids.csv
Saved Fold 3: 165 conversations -> /content/drive/MyDrive/5class_5cv_frozen/fold3_test_ids.csv
Saved Fold 4: 165 conversations -> /content/drive/MyDrive/5class_5cv_frozen/fold4_test_ids.csv
Saved Fold 5: 165 conversations -> /content/drive/MyDrive/5class_5cv_frozen/fold5_test_ids.csv


In [21]:
# ============================================================
# VERIFY SAVED OUTER FOLDS
# ============================================================

saved_outer = {}

for fold_id in range(1, N_FOLDS + 1):

    path = os.path.join(
        SAVE_DIR,
        f"fold{fold_id}_test_ids.csv"
    )

    df = pd.read_csv(path)

    saved_outer[fold_id] = set(
        df["conversation_id"].astype(str)
    )

    print(
        f"Fold {fold_id}: "
        f"{len(saved_outer[fold_id])} IDs"
    )


all_saved_ids = set().union(
    *saved_outer.values()
)

assert len(all_saved_ids) == 826

for i in range(1, 6):
    for j in range(i + 1, 6):
        assert saved_outer[i].isdisjoint(saved_outer[j])

print("\nPASS: frozen outer folds reload correctly.")

Fold 1: 166 IDs
Fold 2: 165 IDs
Fold 3: 165 IDs
Fold 4: 165 IDs
Fold 5: 165 IDs

PASS: frozen outer folds reload correctly.


In [22]:
# ============================================================
# BUILD INTERNAL TRAIN / VAL SPLITS FOR ALL 5 CV RUNS
# ============================================================

from sklearn.model_selection import train_test_split

SEED = 42

all_conv_ids = set(
    eligible_df["conv_id"].astype(str)
)

cv_splits = {}

for cv in range(1, N_FOLDS + 1):

    # ------------------------------------------
    # Outer TEST fold
    # ------------------------------------------
    test_ids = set(
        str(x) for x in folds[cv]
    )

    # ------------------------------------------
    # Remaining conversations = development pool
    # ------------------------------------------
    dev_ids = sorted(
        all_conv_ids - test_ids
    )

    print(f"\n========== CV{cv} ==========")
    print("Development pool:", len(dev_ids))
    print("Test conversations:", len(test_ids))

    # ------------------------------------------
    # Fixed random 80/20 train-val split
    #
    # CV1:
    # 660 -> 528 train / 132 val
    #
    # CV2-5:
    # 661 -> 529 train / 132 val
    # ------------------------------------------
    if len(dev_ids) == 660:
        train_size = 528
        val_size = 132

    elif len(dev_ids) == 661:
        train_size = 529
        val_size = 132

    else:
        raise ValueError(
            f"Unexpected development pool size: {len(dev_ids)}"
        )

    train_ids, val_ids = train_test_split(
        dev_ids,
        train_size=train_size,
        test_size=val_size,
        random_state=SEED,
        shuffle=True
    )

    train_ids = set(train_ids)
    val_ids = set(val_ids)

    # ------------------------------------------
    # Leakage checks
    # ------------------------------------------
    assert train_ids.isdisjoint(val_ids)
    assert train_ids.isdisjoint(test_ids)
    assert val_ids.isdisjoint(test_ids)

    assert (
        train_ids |
        val_ids |
        test_ids
    ) == all_conv_ids

    cv_splits[cv] = {
        "train": train_ids,
        "val": val_ids,
        "test": test_ids,
    }

    print("Train:", len(train_ids))
    print("Val  :", len(val_ids))
    print("Test :", len(test_ids))

    print("Train ∩ Val :", len(train_ids & val_ids))
    print("Train ∩ Test:", len(train_ids & test_ids))
    print("Val ∩ Test  :", len(val_ids & test_ids))

    print(f"PASS: CV{cv} split is conversation-disjoint.")


========== CV1 ==========
Development pool: 660
Test conversations: 166
Train: 528
Val  : 132
Test : 166
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0
PASS: CV1 split is conversation-disjoint.

========== CV2 ==========
Development pool: 661
Test conversations: 165
Train: 529
Val  : 132
Test : 165
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0
PASS: CV2 split is conversation-disjoint.

========== CV3 ==========
Development pool: 661
Test conversations: 165
Train: 529
Val  : 132
Test : 165
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0
PASS: CV3 split is conversation-disjoint.

========== CV4 ==========
Development pool: 661
Test conversations: 165
Train: 529
Val  : 132
Test : 165
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0
PASS: CV4 split is conversation-disjoint.

========== CV5 ==========
Development pool: 661
Test conversations: 165
Train: 529
Val  : 132
Test : 165
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0
PASS: CV5 split is conversation-disjoint.


In [23]:
# ============================================================
# SAVE CV1-CV5 TRAIN / VAL / TEST IDS
# ============================================================

for cv in range(1, N_FOLDS + 1):

    for split_name in ["train", "val", "test"]:

        ids = sorted(
            cv_splits[cv][split_name]
        )

        out_df = pd.DataFrame({
            "conversation_id": ids
        })

        path = os.path.join(
            SAVE_DIR,
            f"cv{cv}_{split_name}_ids.csv"
        )

        out_df.to_csv(
            path,
            index=False
        )

        print(
            f"Saved CV{cv} {split_name}: "
            f"{len(ids)} IDs"
        )

Saved CV1 train: 528 IDs
Saved CV1 val: 132 IDs
Saved CV1 test: 166 IDs
Saved CV2 train: 529 IDs
Saved CV2 val: 132 IDs
Saved CV2 test: 165 IDs
Saved CV3 train: 529 IDs
Saved CV3 val: 132 IDs
Saved CV3 test: 165 IDs
Saved CV4 train: 529 IDs
Saved CV4 val: 132 IDs
Saved CV4 test: 165 IDs
Saved CV5 train: 529 IDs
Saved CV5 val: 132 IDs
Saved CV5 test: 165 IDs


In [24]:
# ============================================================
# FINAL 5-CV SPLIT AUDIT
# ============================================================

for cv in range(1, N_FOLDS + 1):

    train_ids = cv_splits[cv]["train"]
    val_ids   = cv_splits[cv]["val"]
    test_ids  = cv_splits[cv]["test"]

    assert train_ids.isdisjoint(val_ids)
    assert train_ids.isdisjoint(test_ids)
    assert val_ids.isdisjoint(test_ids)

    assert (
        train_ids |
        val_ids |
        test_ids
    ) == all_conv_ids

    print(
        f"CV{cv}: PASS | "
        f"Train={len(train_ids)}, "
        f"Val={len(val_ids)}, "
        f"Test={len(test_ids)}"
    )


# Every conversation must appear in TEST exactly once
test_membership_count = Counter()

for cv in range(1, N_FOLDS + 1):
    for cid in cv_splits[cv]["test"]:
        test_membership_count[cid] += 1

assert len(test_membership_count) == 826
assert all(
    count == 1
    for count in test_membership_count.values()
)

print("\nPASS: every conversation is test exactly once across 5 CV runs.")

CV1: PASS | Train=528, Val=132, Test=166
CV2: PASS | Train=529, Val=132, Test=165
CV3: PASS | Train=529, Val=132, Test=165
CV4: PASS | Train=529, Val=132, Test=165
CV5: PASS | Train=529, Val=132, Test=165

PASS: every conversation is test exactly once across 5 CV runs.


In [25]:
# ============================================================
# ORIGINAL TRAIN / VAL / TEST CLASS DISTRIBUTION — ALL 5 CVs
# ============================================================

summary_rows = []

for cv in range(1, N_FOLDS + 1):

    train_ids = cv_splits[cv]["train"]
    val_ids   = cv_splits[cv]["val"]
    test_ids  = cv_splits[cv]["test"]

    for split_name, split_ids in [
        ("Train", train_ids),
        ("Val", val_ids),
        ("Test", test_ids),
    ]:

        part = eligible_df[
            eligible_df["conv_id"].astype(str).isin(split_ids)
        ]

        counts = {
            intent: int(part[intent].sum())
            for intent in TARGET_INTENTS
        }

        summary_rows.append({
            "CV": f"CV{cv}",
            "Split": split_name,
            "Conversations": len(split_ids),
            **counts,
            "Total_targets": sum(counts.values()),
        })

distribution_df = pd.DataFrame(summary_rows)

display(distribution_df)

print("\nTRAIN distributions only:")
display(
    distribution_df[
        distribution_df["Split"] == "Train"
    ].reset_index(drop=True)
)

,CV,Split,Conversations,Reveal,Interpret,Revise,Chitchat,Inquire,Total_targets
0,CV1,Train,528,660,1147,39,173,11,2030
1,CV1,Val,132,164,303,15,42,2,526
2,CV1,Test,166,207,363,13,53,4,640
3,CV2,Train,529,653,1163,46,174,10,2046
4,CV2,Val,132,172,288,8,40,3,511
5,CV2,Test,165,206,362,13,54,4,639
6,CV3,Train,529,674,1145,45,166,12,2042
7,CV3,Val,132,151,306,8,49,2,516
8,CV3,Test,165,206,362,14,53,3,638
9,CV4,Train,529,669,1164,40,177,13,2063



TRAIN distributions only:


,CV,Split,Conversations,Reveal,Interpret,Revise,Chitchat,Inquire,Total_targets
0,CV1,Train,528,660,1147,39,173,11,2030
1,CV2,Train,529,653,1163,46,174,10,2046
2,CV3,Train,529,674,1145,45,166,12,2042
3,CV4,Train,529,669,1164,40,177,13,2063
4,CV5,Train,529,658,1134,49,169,14,2024


In [26]:
# ============================================================
# FINAL SAVED-ID AUDIT BEFORE AUGMENTATION
# ============================================================

from collections import Counter
import os
import pandas as pd

all_conv_ids = set(
    eligible_df["conv_id"].astype(str)
)

saved_cv = {}

for cv in range(1, 6):

    saved_cv[cv] = {}

    for split in ["train", "val", "test"]:

        path = os.path.join(
            SAVE_DIR,
            f"cv{cv}_{split}_ids.csv"
        )

        df = pd.read_csv(path)

        ids = set(
            df["conversation_id"].astype(str)
        )

        saved_cv[cv][split] = ids

    train_ids = saved_cv[cv]["train"]
    val_ids   = saved_cv[cv]["val"]
    test_ids  = saved_cv[cv]["test"]

    # Pairwise disjoint
    assert train_ids.isdisjoint(val_ids)
    assert train_ids.isdisjoint(test_ids)
    assert val_ids.isdisjoint(test_ids)

    # Complete coverage
    assert (
        train_ids | val_ids | test_ids
    ) == all_conv_ids

    # Test must exactly equal frozen outer fold
    frozen_test_ids = set(
        str(x) for x in folds[cv]
    )

    assert test_ids == frozen_test_ids

    print(
        f"CV{cv}: PASS | "
        f"Train={len(train_ids)}, "
        f"Val={len(val_ids)}, "
        f"Test={len(test_ids)}"
    )


# ------------------------------------------------------------
# Test-fold checks across CV runs
# ------------------------------------------------------------

for i in range(1, 6):
    for j in range(i + 1, 6):

        assert saved_cv[i]["test"].isdisjoint(
            saved_cv[j]["test"]
        )


# Every conversation appears in test exactly once
test_counter = Counter()

for cv in range(1, 6):
    for cid in saved_cv[cv]["test"]:
        test_counter[cid] += 1

assert len(test_counter) == 826
assert all(v == 1 for v in test_counter.values())

print("\nPASS: all saved CV files are frozen, disjoint,")
print("match the outer folds, and every conversation")
print("appears in the held-out test set exactly once.")

CV1: PASS | Train=528, Val=132, Test=166
CV2: PASS | Train=529, Val=132, Test=165
CV3: PASS | Train=529, Val=132, Test=165
CV4: PASS | Train=529, Val=132, Test=165
CV5: PASS | Train=529, Val=132, Test=165

PASS: all saved CV files are frozen, disjoint,
match the outer folds, and every conversation
appears in the held-out test set exactly once.
